# Inspect One Transformer Batch - v5

This notebook loads one provider-built 1m-bar batch, checks tensor shapes and finite values, loads a model checkpoint if provided, runs one inference, and computes one Smooth L1 loss.

In [ ]:
from pathlib import Path

MODEL_VERSION = "v5"
MODEL_ROOT = Path(r"D:\\TradingData\\quant-research-workbench\\market_data\\models\\inhouse_transformer") / MODEL_VERSION

# Edit these values for the batch/checkpoint you want to inspect.
WORKSPACE = Path(r"D:\\TradingCodes\\quant-research-workbench")
SESSION_DATE = "2024-01-22"
TICKERS = ("USO",)  # Use a small tuple first. Example: ("USO", "CADL")
BATCH_SIZE = 16
CONTEXT_LENGTH = 64
HORIZON = 1
TARGET_COLUMNS = ("close",)
TARGET_MODE = "return_bps"
SESSION_SCOPE = "all"
DEVICE = "cuda"  # Use "cpu" if CUDA is unavailable.
USE_AMP = False  # Keep False while checking exact tensors/losses.
LOAD_CHECKPOINT_CONFIG = True  # Use checkpoint model/data shape when CHECKPOINT_PATH is set.

# Paste a checkpoint path, or leave empty to auto-load the latest v5 last.pt/best.pt under MODEL_ROOT.
CHECKPOINT_PATH = ""
# Example:
# CHECKPOINT_PATH = str(MODEL_ROOT / "feature_temporal_v5_return_bps_ctx64_h1_..." / "last.pt")

In [ ]:
import sys
import math
from dataclasses import fields
import numpy as np
import polars as pl
import torch

if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from research.inhouse_transformer.v5.config import DataConfig, ModelConfig, ExperimentConfig
from research.inhouse_transformer.v5.data import RollingBarWindowDataset, available_sessions, target_values_to_bps
from research.inhouse_transformer.v5.model import FeatureTemporalTransformer, forecast_loss

np.set_printoptions(precision=5, suppress=True)
torch.set_printoptions(precision=5, sci_mode=False)

def dataclass_from_dict(cls, payload, tuple_fields=()):
    allowed = {field.name for field in fields(cls)}
    values = {key: value for key, value in payload.items() if key in allowed}
    if cls is DataConfig and "processed_root" in values:
        values["processed_root"] = Path(values["processed_root"])
    for name in tuple_fields:
        if name in values:
            values[name] = tuple(values[name])
    return cls(**values)

def latest_checkpoint(model_root):
    candidates = list(model_root.glob("*/last.pt")) + list(model_root.glob("*/best.pt"))
    return max(candidates, key=lambda path: path.stat().st_mtime) if candidates else None

checkpoint = None
checkpoint_path = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else latest_checkpoint(MODEL_ROOT)
if checkpoint_path:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    checkpoint_config = checkpoint.get("config", {}) if isinstance(checkpoint, dict) else {}
else:
    checkpoint_config = {}
checkpoint_version = checkpoint_config.get("experiment_version") if checkpoint_config else None
if checkpoint_version and checkpoint_version != MODEL_VERSION:
    raise ValueError(f"Checkpoint version {checkpoint_version!r} does not match notebook version {MODEL_VERSION!r}.")

if checkpoint_config and LOAD_CHECKPOINT_CONFIG:
    data_config = dataclass_from_dict(
        DataConfig,
        checkpoint_config.get("data", {}),
        tuple_fields=("target_columns", "input_feature_columns", "time_feature_columns", "tickers"),
    )
    model_config = dataclass_from_dict(ModelConfig, checkpoint_config.get("model", {}))
    print("using data/model shape from checkpoint config")
else:
    data_config = DataConfig(
        context_length=CONTEXT_LENGTH,
        horizon=HORIZON,
        target_mode=TARGET_MODE,
        target_columns=TARGET_COLUMNS,
        input_normalization="window_zscore_only",
    )
    model_config = ModelConfig()

# These are inspection controls, so they intentionally override checkpoint split/ticker settings.
data_config.train_start_date = SESSION_DATE
data_config.train_end_date = SESSION_DATE
data_config.validation_start_date = SESSION_DATE
data_config.validation_end_date = SESSION_DATE
data_config.test_start_date = SESSION_DATE
data_config.test_end_date = SESSION_DATE
data_config.session_scope = SESSION_SCOPE
data_config.tickers = TICKERS
data_config.max_tickers = len(TICKERS)
experiment_config = ExperimentConfig(data=data_config, model=model_config)

device = torch.device(DEVICE if DEVICE == "cpu" or torch.cuda.is_available() else "cpu")
print(f"device={device}")
print(f"input_features={len(data_config.input_feature_columns)} {data_config.input_feature_columns}")
print(f"context={data_config.context_length} horizon={data_config.horizon} targets={data_config.target_columns}")
print(f"target_mode={data_config.target_mode} input_normalization={data_config.input_normalization}")
print(f"time_features={len(data_config.time_feature_columns)} {data_config.time_feature_columns}")
print(f"targets={data_config.target_columns} horizon={data_config.horizon} target_mode={data_config.target_mode}")
print(f"input_normalization={data_config.input_normalization}")

In [ ]:
sessions = available_sessions(data_config.processed_root, SESSION_DATE, SESSION_DATE)
dataset = RollingBarWindowDataset(
    config=data_config,
    sessions=sessions,
    tickers=TICKERS,
    batch_size=BATCH_SIZE,
    seed=17,
    mode="inspect",
    max_windows=BATCH_SIZE,
    shuffle=False,
)

batch = next(iter(dataset))
print("batch keys:", sorted(batch.keys()))
for key, value in batch.items():
    if torch.is_tensor(value):
        finite = bool(torch.isfinite(value).all())
        min_value = float(value.min()) if value.numel() else math.nan
        max_value = float(value.max()) if value.numel() else math.nan
        print(f"{key:20s} shape={tuple(value.shape)} dtype={value.dtype} finite={finite} min={min_value:.6g} max={max_value:.6g}")
    else:
        print(f"{key:20s} type={type(value).__name__} len={len(value) if hasattr(value, '__len__') else 'n/a'} sample={value[:5] if isinstance(value, list) else value}")

In [ ]:
batch["values"]

In [ ]:
# Inspect the first sample's latest context bar after input normalization.
feature_names = list(data_config.input_feature_columns)
time_feature_names = list(data_config.time_feature_columns)
last_values = batch["values"][0, -1].numpy()
last_time_features = batch["time_features"][0, -1].numpy()

print("first sample ticker:", batch.get("ticker", [None])[0])
print("current_close:", float(batch["current_close"][0]))
print("target_center:", float(batch["target_center"][0]), "target_scale:", float(batch["target_scale"][0]))

pl.DataFrame({"feature": feature_names, "normalized_value_at_context_end": last_values}).head(20)

In [ ]:
model = FeatureTemporalTransformer(
    feature_count=len(data_config.input_feature_columns),
    time_feature_count=len(data_config.time_feature_columns),
    context_length=data_config.context_length,
    horizon=data_config.horizon,
    target_count=len(data_config.target_columns),
    config=model_config,
).to(device)

if checkpoint is not None:
    state_dict = checkpoint.get("model", checkpoint)
    model.load_state_dict(state_dict, strict=True)
    print(f"loaded checkpoint: {checkpoint_path}")
    print("checkpoint step:", checkpoint.get("step"), "best_score:", checkpoint.get("best_score"))
else:
    print("CHECKPOINT_PATH is empty; using fresh randomly initialized model.")

param_count = sum(parameter.numel() for parameter in model.parameters())
print(f"model parameters={param_count:,}")

In [ ]:
# Model architecture diagram with tensor shapes.
import math
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch


def _active_data_config():
    if "data_config" in globals():
        return data_config
    if "config" in globals() and hasattr(config, "data"):
        return config.data
    raise RuntimeError("Run the config-loading cells before drawing the architecture diagram.")


def _active_model_config():
    if "model_config" in globals():
        return model_config
    if "config" in globals() and hasattr(config, "model"):
        return config.model
    raise RuntimeError("Run the model-config cells before drawing the architecture diagram.")


def _tensor_shape(name, fallback):
    current_batch = globals().get("batch")
    if isinstance(current_batch, dict) and name in current_batch:
        return tuple(current_batch[name].shape)
    return fallback


def _box(ax, x, y, w, h, text, *, fc="#f7f9fc", ec="#365f91", fontsize=9):
    patch = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.02,rounding_size=0.018",
        linewidth=1.2,
        edgecolor=ec,
        facecolor=fc,
    )
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fontsize, wrap=True)
    return patch


def _arrow(ax, start, end):
    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle="-|>",
            mutation_scale=12,
            linewidth=1.0,
            color="#333333",
            shrinkA=4,
            shrinkB=4,
        )
    )


def _single_branch_diagram(data_cfg, model_cfg):
    feature_count = len(data_cfg.input_feature_columns)
    time_columns = tuple(getattr(data_cfg, "time_feature_columns", ()))
    time_count = len(time_columns)
    context_length = data_cfg.context_length
    horizon = data_cfg.horizon
    target_count = len(data_cfg.target_columns)
    values_shape = _tensor_shape("values", ("B", context_length, feature_count))
    time_shape = _tensor_shape("time_features", ("B", context_length, time_count)) if time_count else None

    fig, ax = plt.subplots(figsize=(18, 7))
    ax.set_axis_off()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    fig.suptitle(f"{MODEL_VERSION} Feature-Temporal Transformer", fontsize=16, fontweight="bold")

    _box(ax, 0.03, 0.62, 0.18, 0.16, f"values\n{values_shape}")
    if time_shape is not None:
        _box(ax, 0.03, 0.38, 0.18, 0.16, f"time_features\n{time_shape}", fc="#f3fbf6", ec="#3b7f4c")
        embed_text = "value projection + feature embedding\n+ position embedding + time projection"
        _arrow(ax, (0.21, 0.70), (0.28, 0.58))
        _arrow(ax, (0.21, 0.46), (0.28, 0.58))
    else:
        embed_text = "value projection + feature embedding\n+ position embedding\ntime already encoded in value channels"
        _arrow(ax, (0.21, 0.70), (0.28, 0.58))

    _box(ax, 0.28, 0.48, 0.18, 0.20, embed_text, fc="#fff8ed", ec="#b7791f")
    _box(
        ax,
        0.51,
        0.61,
        0.17,
        0.15,
        "feature Transformer\nattention across features\n"
        f"layers={model_cfg.feature_attention_layers}",
        fc="#eef6ff",
        ec="#2b6cb0",
    )
    _box(ax, 0.51, 0.41, 0.17, 0.12, "softmax feature pooling\nper bar -> [B, context, d_model]", fc="#f7f7ff", ec="#5a67d8")
    _box(
        ax,
        0.73,
        0.50,
        0.17,
        0.18,
        "temporal Transformer\nattention across bars\n"
        f"layers={model_cfg.temporal_layers}",
        fc="#eefaf7",
        ec="#2c7a7b",
    )
    _box(ax, 0.73, 0.25, 0.17, 0.12, "last token + LayerNorm\n[B, d_model]", fc="#f5f0ff", ec="#6b46c1")
    _box(ax, 0.93, 0.57, 0.06, 0.12, f"regression\n[B,{horizon},{target_count}]")
    _box(ax, 0.93, 0.33, 0.06, 0.12, f"direction\n[B,{horizon}]", fc="#fff5f5", ec="#c53030")

    _arrow(ax, (0.46, 0.58), (0.51, 0.68))
    _arrow(ax, (0.595, 0.61), (0.595, 0.53))
    _arrow(ax, (0.68, 0.47), (0.73, 0.59))
    _arrow(ax, (0.815, 0.50), (0.815, 0.37))
    _arrow(ax, (0.90, 0.31), (0.93, 0.39))
    _arrow(ax, (0.90, 0.31), (0.93, 0.63))

    ax.text(
        0.03,
        0.08,
        f"d_model={model_cfg.d_model} | heads={model_cfg.num_heads} | ff_dim={model_cfg.ff_dim} | "
        f"features={feature_count} | time_features={time_count}",
        fontsize=10,
        ha="left",
        va="center",
    )
    plt.show()


def _multiscale_diagram(data_cfg, model_cfg):
    feature_count = len(data_cfg.input_feature_columns)
    time_count = len(data_cfg.time_feature_columns)
    five_len = math.ceil(data_cfg.five_min_lookback_minutes / data_cfg.five_min_bucket_minutes)
    thirty_len = math.ceil(data_cfg.thirty_min_lookback_minutes / data_cfg.thirty_min_bucket_minutes)
    multiscale_feature_count = len(globals().get("MULTISCALE_FEATURE_COLUMNS", getattr(globals().get("train_mod", object()), "MULTISCALE_FEATURE_COLUMNS", tuple(range(15)))))
    anchor_feature_count = len(globals().get("ANCHOR_VALUE_COLUMNS", getattr(globals().get("train_mod", object()), "ANCHOR_VALUE_COLUMNS", tuple(range(11)))))
    anchor_count = len(globals().get("ANCHOR_NAMES", getattr(globals().get("train_mod", object()), "ANCHOR_NAMES", tuple(range(19)))))
    horizon = data_cfg.horizon
    target_count = len(data_cfg.target_columns)

    branches = [
        ("1m", _tensor_shape("values", ("B", data_cfg.context_length, feature_count)), _tensor_shape("time_features", ("B", data_cfg.context_length, time_count)), "last 64 1m bars"),
        ("5m", _tensor_shape("five_min_values", ("B", five_len, multiscale_feature_count)), _tensor_shape("five_min_time_features", ("B", five_len, time_count)), "5h aggregated into 5m buckets"),
        ("30m", _tensor_shape("thirty_min_values", ("B", thirty_len, multiscale_feature_count)), _tensor_shape("thirty_min_time_features", ("B", thirty_len, time_count)), "24h aggregated into 30m buckets"),
        ("anchors", _tensor_shape("anchor_values", ("B", anchor_count, anchor_feature_count)), _tensor_shape("anchor_time_features", ("B", anchor_count, time_count)), "same-minute + day/week summary anchors"),
    ]

    fig, ax = plt.subplots(figsize=(20, 10))
    ax.set_axis_off()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    fig.suptitle(f"{MODEL_VERSION} Multiscale Feature-Temporal Transformer", fontsize=16, fontweight="bold")

    y_positions = [0.78, 0.58, 0.38, 0.18]
    summary_points = []
    for (name, value_shape, time_shape, description), y in zip(branches, y_positions):
        _box(ax, 0.02, y, 0.12, 0.12, f"{name} values\n{value_shape}")
        _box(ax, 0.02, y - 0.12, 0.12, 0.09, f"time\n{time_shape}", fc="#f3fbf6", ec="#3b7f4c")
        _box(ax, 0.18, y - 0.04, 0.18, 0.12, f"{name} embedding\nvalue + feature + position + time\n{description}", fc="#fff8ed", ec="#b7791f", fontsize=8)
        _box(ax, 0.40, y - 0.04, 0.16, 0.12, f"feature Transformer\nfeatures inside token\nlayers={model_cfg.feature_attention_layers}", fc="#eef6ff", ec="#2b6cb0", fontsize=8)
        _box(ax, 0.60, y - 0.04, 0.15, 0.12, "feature pool\nthen temporal Transformer\n" f"layers={model_cfg.temporal_layers}", fc="#eefaf7", ec="#2c7a7b", fontsize=8)
        _box(ax, 0.79, y - 0.02, 0.10, 0.08, f"{name} summary\n[B,d_model]", fc="#f5f0ff", ec="#6b46c1", fontsize=8)
        _arrow(ax, (0.14, y + 0.04), (0.18, y + 0.02))
        _arrow(ax, (0.14, y - 0.075), (0.18, y + 0.02))
        _arrow(ax, (0.36, y + 0.02), (0.40, y + 0.02))
        _arrow(ax, (0.56, y + 0.02), (0.60, y + 0.02))
        _arrow(ax, (0.75, y + 0.02), (0.79, y + 0.02))
        summary_points.append((0.89, y + 0.02))

    _box(ax, 0.91, 0.55, 0.08, 0.16, "stack 4 summaries\n+ scale embedding\n[B,4,d_model]", fc="#fffaf0", ec="#b7791f", fontsize=8)
    _box(ax, 0.91, 0.35, 0.08, 0.13, "fusion Transformer\n1 layer\nmean pool + LN", fc="#edf2ff", ec="#5a67d8", fontsize=8)
    _box(ax, 0.91, 0.17, 0.08, 0.10, f"regression\n[B,{horizon},{target_count}]", fontsize=8)
    _box(ax, 0.91, 0.04, 0.08, 0.10, f"direction\n[B,{horizon}]", fc="#fff5f5", ec="#c53030", fontsize=8)
    for point in summary_points:
        _arrow(ax, point, (0.91, 0.63))
    _arrow(ax, (0.95, 0.55), (0.95, 0.48))
    _arrow(ax, (0.95, 0.35), (0.95, 0.27))
    _arrow(ax, (0.95, 0.35), (0.95, 0.14))

    ax.text(
        0.02,
        0.03,
        f"d_model={model_cfg.d_model} | heads={model_cfg.num_heads} | ff_dim={model_cfg.ff_dim} | "
        f"time_features={time_count} | branch encoders do not share weights",
        fontsize=10,
        ha="left",
        va="center",
    )
    plt.show()


def draw_model_architecture():
    data_cfg = _active_data_config()
    model_cfg = _active_model_config()
    if hasattr(data_cfg, "five_min_lookback_minutes"):
        _multiscale_diagram(data_cfg, model_cfg)
    else:
        _single_branch_diagram(data_cfg, model_cfg)


draw_model_architecture()


In [ ]:
def move_tensor_batch(batch, device):
    return {key: value.to(device, non_blocking=True) if torch.is_tensor(value) else value for key, value in batch.items()}

model.eval()
device_batch = move_tensor_batch(batch, device)
with torch.inference_mode():
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP and device.type == "cuda"):
        prediction, direction_logits = model(device_batch["values"], device_batch["time_features"])
        loss, loss_parts = forecast_loss(
            prediction,
            device_batch["targets"],
            direction_logits,
            device_batch["direction"],
            model_config.direction_loss_weight,
        )

print("prediction shape:", tuple(prediction.shape))
print("direction_logits shape:", tuple(direction_logits.shape))
print("loss:", float(loss.detach().cpu()))
print("loss_parts:", loss_parts)
print("prediction finite:", bool(torch.isfinite(prediction).all()))
print("target finite:", bool(torch.isfinite(device_batch["targets"]).all()))

In [ ]:
def prediction_and_target_prices(prediction_tensor, batch_dict, data_config):
    pred_np = prediction_tensor.detach().cpu().numpy()
    target_np = batch_dict["targets"].detach().cpu().numpy()
    current_close = batch_dict["current_close"].detach().cpu().numpy()
    if data_config.target_mode == "actual_price_zscore":
        center = batch_dict["target_center"].detach().cpu().numpy().reshape(-1, 1, 1)
        scale = batch_dict["target_scale"].detach().cpu().numpy().reshape(-1, 1, 1)
        return pred_np * scale + center, target_np * scale + center
    if data_config.target_mode == "return_bps":
        current = np.maximum(current_close.reshape(-1, 1, 1), 1e-6)
        return current * np.exp(pred_np / 10000.0), current * np.exp(target_np / 10000.0)
    raise ValueError(data_config.target_mode)

pred_prices, target_prices = prediction_and_target_prices(prediction, device_batch, data_config)
pred_bps = target_values_to_bps(
    prediction.detach().cpu().numpy(),
    batch["current_close"].numpy(),
    batch["target_center"].numpy(),
    batch["target_scale"].numpy(),
    data_config.target_mode,
)
target_bps = target_values_to_bps(
    batch["targets"].numpy(),
    batch["current_close"].numpy(),
    batch["target_center"].numpy(),
    batch["target_scale"].numpy(),
    data_config.target_mode,
)

close_index = data_config.target_columns.index("close")
rows = []
row_count = min(len(batch.get("ticker", [])), pred_bps.shape[0], 20)
for idx in range(row_count):
    rows.append({
        "row": idx,
        "ticker": batch.get("ticker", [None] * BATCH_SIZE)[idx],
        "target_timestamp_ns": int(batch["target_timestamp_ns"][idx]),
        "current_close": float(batch["current_close"][idx]),
        "target_close": float(target_prices[idx, 0, close_index]),
        "prediction_close": float(pred_prices[idx, 0, close_index]),
        "target_bps": float(target_bps[idx, 0, close_index]),
        "prediction_bps": float(pred_bps[idx, 0, close_index]),
        "abs_error_bps": abs(float(pred_bps[idx, 0, close_index] - target_bps[idx, 0, close_index])),
        "dir_correct": bool((pred_bps[idx, 0, close_index] > 0.0) == (target_bps[idx, 0, close_index] > 0.0)),
    })

pl.DataFrame(rows)